# Revision - Level 2 - 19/04/2026
- Revision about Pandas and Dataframes basics
- 3. Key Operations

Once grouped, you can perform various operations using the `DataFrameGroupBy` object:

| Operation Type | Description | Common Methods |
| :--- | :--- | :--- |
| **Aggregation** | Reduces each group to a single value. | `.sum()`, `.mean()`, `.count()`, `.max()`, `.min()` |
| **Transformation** | Modifies group values while keeping the original shape. | `.transform()` (e.g., filling missing values with group means) |
| **Filtering** | Discards groups based on a boolean condition. | `.filter()` |
| **Custom Agg** | Applies multiple or specific functions to columns. | `.agg({'Salary': 'mean', 'Bonus': 'sum'})` |

In [91]:
import pandas as pd

## 🔥 MISSÃO 1 — Limpeza de dados

Antes de qualquer análise:

👉 Identifique:

- valores nulos
- colunas problemáticas

Depois:

👉 trate os valores nulos de imdb_rating

💡 Decide:

- remover?
- preencher?
- ignorar?

In [92]:
df = pd.read_csv("movies.csv")
df.head(5)

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.0,100000.0,Thousands,INR,Bengali
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.0,954.8,Millions,USD,English
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English


In [93]:
df.shape

(37, 10)

In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         37 non-null     str    
 1   industry      37 non-null     str    
 2   release_year  37 non-null     int64  
 3   imdb_rating   36 non-null     float64
 4   studio        34 non-null     str    
 5   budget        37 non-null     float64
 6   revenue       37 non-null     float64
 7   unit          37 non-null     str    
 8   currency      37 non-null     str    
 9   language      37 non-null     str    
dtypes: float64(3), int64(1), str(6)
memory usage: 3.0 KB


In [95]:
df_null = df[df.isna().any(axis=1)]
df_null

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
16,Parasite,Hollywood,2019,8.5,NaN,15.5,263.1,Millions,USD,English
24,Bajirao Mastani,Bollywood,2015,7.2,NaN,1.4,3.5,Billions,INR,Hindi
25,Taare Zameen Par,Bollywood,2007,8.3,NaN,120.0,1350.0,Millions,INR,Hindi
28,Sanju,Bollywood,2018,NaN,Vinod Chopra Films,1.0,5.9,Billions,INR,Hindi


In [97]:
column_mean = df["imdb_rating"].mean()
df.fillna({"imdb_rating": column_mean}, inplace = True)
df['imdb_rating'] = df['imdb_rating'].round(1)
df

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.00,954.8,Millions,USD,English
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.00,644.8,Millions,USD,English
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.00,854.0,Millions,USD,English
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.00,670.0,Millions,USD,English
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English
7,The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.00,307.1,Millions,USD,English
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.00,2202.0,Millions,USD,English


In [98]:
df.fillna({"studio": "Unkown"}, inplace = True)
df

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.00,954.8,Millions,USD,English
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.00,644.8,Millions,USD,English
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.00,854.0,Millions,USD,English
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.00,670.0,Millions,USD,English
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English
7,The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.00,307.1,Millions,USD,English
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.00,2202.0,Millions,USD,English


## 🔥 MISSÃO 2 — Padronização (nível importante)

A coluna:

👉 unit (Millions, Billions, Thousands)

Está inconsistente.

👉 Crie uma nova coluna:

revenue_usd (ou padrão único)

💡 Ideia:

transformar tudo para uma mesma escala

In [99]:
def convert_to_billions_usd(row, column):
    value = row[column]

    if row["currency"] == "INR":
        value = value*0.011

    if row["unit"] == "Thousands":
        value = value/1_000_000
    elif row["unit"] == "Millions":
        value = value/1_000
    return value

df["revenue_USD"] = df.apply(lambda x: convert_to_billions_usd(x, 'revenue'), axis=1)
df

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali,0.001100
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.00,954.8,Millions,USD,English,0.954800
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.00,644.8,Millions,USD,English,0.644800
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.00,854.0,Millions,USD,English,0.854000
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.00,670.0,Millions,USD,English,0.670000
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English,0.073300
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English,0.701800
7,The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.00,307.1,Millions,USD,English,0.307100
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English,0.460500
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.00,2202.0,Millions,USD,English,2.202000


## 🔥 MISSÃO 3 — Análise de lucro REAL

Agora que os dados estão padronizados:

👉 Crie:

coluna profit REAL (com base na unidade corrigida)

Depois:

👉 descubra:

- top 5 filmes mais lucrativos
- bottom 5

In [100]:
df["budget_USD"] = df.apply(lambda x: convert_to_billions_usd(x, 'budget'), axis=1)
df["profit_USD"] = df["revenue_USD"] - df["budget_USD"] 
df

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali,0.001100,0.00077,0.000330
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.00,954.8,Millions,USD,English,0.954800,0.20000,0.754800
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.00,644.8,Millions,USD,English,0.644800,0.16500,0.479800
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.00,854.0,Millions,USD,English,0.854000,0.18000,0.674000
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.00,670.0,Millions,USD,English,0.670000,0.25000,0.420000
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English,0.073300,0.02500,0.048300
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English,0.701800,0.16500,0.536800
7,The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.00,307.1,Millions,USD,English,0.307100,0.05500,0.252100
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English,0.460500,0.10300,0.357500
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.00,2202.0,Millions,USD,English,2.202000,0.20000,2.002000


In [101]:
df.nlargest(5, 'profit_USD')

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
11,Avatar,Hollywood,2009,7.8,20th Century Fox,237.0,2847.0,Millions,USD,English,2.847,0.237,2.610
17,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English,2.798,0.400,2.398
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.0,2202.0,Millions,USD,English,2.202,0.200,2.002
18,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English,2.048,0.400,1.648
15,Jurassic Park,Hollywood,1993,8.2,Universal Pictures,63.0,1046.0,Millions,USD,English,1.046,0.063,0.983


In [102]:
df.nsmallest(5, 'profit_USD')

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
10,It's a Wonderful Life,Hollywood,1946,8.6,Liberty Films,3.18,3.3,Millions,USD,English,0.00330,0.00318,0.00012
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali,0.00110,0.00077,0.00033
26,Munna Bhai M.B.B.S.,Bollywood,2003,8.1,Vinod Chopra Productions,100.00,410.0,Millions,INR,Hindi,0.00451,0.00110,0.00341
32,Shershaah,Bollywood,2021,8.4,Dharma Productions,500.00,950.0,Millions,INR,Hindi,0.01045,0.00550,0.00495
23,Kabhi Khushi Kabhie Gham,Bollywood,2001,7.4,Dharma Productions,390.00,1360.0,Millions,INR,Hindi,0.01496,0.00429,0.01067


## 🔥 MISSÃO 4 — Análise por indústria

Compare:

👉 Hollywood vs Bollywood

Descubra:

- média de rating
- média de lucro
- quantidade de filmes

💡 Aqui entra:
`groupby`

In [103]:
# Comparasion without .agg()
print(df.groupby("industry")["imdb_rating"].mean())
print(df.groupby("industry")["profit_USD"].mean())
print(df.groupby("industry")["title"].count())

industry
Bollywood    7.670588
Hollywood    8.130000
Name: imdb_rating, dtype: float64
industry
Bollywood    0.040298
Hollywood    0.775416
Name: profit_USD, dtype: float64
industry
Bollywood    17
Hollywood    20
Name: title, dtype: int64


In [104]:
# Comparasion with .agg()
industry_comparasion = df.groupby("industry").agg({
    "imdb_rating": "mean",
    "profit_USD": "mean",
    "title": "count"
}) 
print(industry_comparasion)

           imdb_rating  profit_USD  title
industry                                 
Bollywood     7.670588    0.040298     17
Hollywood     8.130000    0.775416     20


## 🔥 MISSÃO 5 — Análise por linguagem

👉 Descubra:

- média de rating por language
- linguagem com maior média

In [105]:
print(df.groupby("language")["imdb_rating"].mean())
print(df.groupby("language")["imdb_rating"].mean().idxmax())

language
Bengali    8.300000
English    8.130000
Hindi      7.508333
Kannada    8.400000
Telugu     7.866667
Name: imdb_rating, dtype: float64
Kannada


## 🔥 MISSÃO 6 — Filtro inteligente

👉 Encontre:

Filmes que:

- têm rating acima da média
- E lucro acima da média

💡 Esse é nível intermediário real

In [106]:
df_above_avg = df[(df["imdb_rating"] > df.imdb_rating.mean()) & (df["profit_USD"] > df.profit_USD.mean())]
df_above_avg

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.0,701.8,Millions,USD,English,0.7018,0.165,0.5368
13,The Dark Knight,Hollywood,2008,9.0,Syncopy,185.0,1006.0,Millions,USD,English,1.0060,0.185,0.8210
15,Jurassic Park,Hollywood,1993,8.2,Universal Pictures,63.0,1046.0,Millions,USD,English,1.0460,0.063,0.9830
17,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English,2.7980,0.400,2.3980
18,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English,2.0480,0.400,1.6480


## 🔥 MISSÃO 7 — Insights de negócio

Responda:

👉 “Filmes com maior budget geram mais lucro?”

💡 Dica:

- comparar médias
- ou observar padrões

In [117]:
df.nlargest(10, 'profit_USD')

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
11,Avatar,Hollywood,2009,7.8,20th Century Fox,237.0,2847.0,Millions,USD,English,2.8470,0.237,2.6100
17,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English,2.7980,0.400,2.3980
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.0,2202.0,Millions,USD,English,2.2020,0.200,2.0020
18,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English,2.0480,0.400,1.6480
15,Jurassic Park,Hollywood,1993,8.2,Universal Pictures,63.0,1046.0,Millions,USD,English,1.0460,0.063,0.9830
13,The Dark Knight,Hollywood,2008,9.0,Syncopy,185.0,1006.0,Millions,USD,English,1.0060,0.185,0.8210
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.0,954.8,Millions,USD,English,0.9548,0.200,0.7548
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English,0.8540,0.180,0.6740
20,Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English,0.7144,0.177,0.5374
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.0,701.8,Millions,USD,English,0.7018,0.165,0.5368


In [116]:
df.nlargest(10, 'imdb_rating')

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English,0.0733,0.02500,0.04830
12,The Godfather,Hollywood,1972,9.2,Paramount Pictures,7.20,291.0,Millions,USD,English,0.2910,0.00720,0.28380
13,The Dark Knight,Hollywood,2008,9.0,Syncopy,185.00,1006.0,Millions,USD,English,1.0060,0.18500,0.82100
14,Schindler's List,Hollywood,1993,9.0,Universal Pictures,22.00,322.2,Millions,USD,English,0.3222,0.02200,0.30020
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English,0.7018,0.16500,0.53680
10,It's a Wonderful Life,Hollywood,1946,8.6,Liberty Films,3.18,3.3,Millions,USD,English,0.0033,0.00318,0.00012
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English,0.4605,0.10300,0.35750
16,Parasite,Hollywood,2019,8.5,Unkown,15.50,263.1,Millions,USD,English,0.2631,0.01550,0.24760
17,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.00,2798.0,Millions,USD,English,2.7980,0.40000,2.39800
18,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.00,2048.0,Millions,USD,English,2.0480,0.40000,1.64800


In [115]:
df.nlargest(10, 'budget_USD')

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD
17,Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English,2.7980,0.4000,2.3980
18,Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English,2.0480,0.4000,1.6480
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English,0.6700,0.2500,0.4200
11,Avatar,Hollywood,2009,7.8,20th Century Fox,237.0,2847.0,Millions,USD,English,2.8470,0.2370,2.6100
19,Captain America: The First Avenger,Hollywood,2011,6.9,Marvel Studios,216.7,370.6,Millions,USD,English,0.3706,0.2167,0.1539
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.0,954.8,Millions,USD,English,0.9548,0.2000,0.7548
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.0,2202.0,Millions,USD,English,2.2020,0.2000,2.0020
13,The Dark Knight,Hollywood,2008,9.0,Syncopy,185.0,1006.0,Millions,USD,English,1.0060,0.1850,0.8210
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English,0.8540,0.1800,0.6740
20,Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English,0.7144,0.1770,0.5374


## 🔥 MISSÃO 8 — Criação de categorias

👉 Crie uma coluna:

rating_category

Regras:

- 8 → Excellent
- 7–8 → Good
- < 7 → Bad

In [119]:
def categorize(row):
    if row["imdb_rating"] > 8.0:
        return "Excellent"
    elif row["imdb_rating"] >= 7.0 and row["imdb_rating"] <= 8.0:
        return "Good"
    elif row["imdb_rating"] < 7:
        return "Bad"

df["rating_category"] = df.apply(categorize, axis=1)
df

,title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language,revenue_USD,budget_USD,profit_USD,rating_category
0,Pather Panchali,Bollywood,1955,8.3,Government of West Bengal,70000.00,100000.0,Thousands,INR,Bengali,0.001100,0.00077,0.000330,Excellent
1,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,200.00,954.8,Millions,USD,English,0.954800,0.20000,0.754800,Good
2,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.00,644.8,Millions,USD,English,0.644800,0.16500,0.479800,Bad
3,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.00,854.0,Millions,USD,English,0.854000,0.18000,0.674000,Good
4,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.00,670.0,Millions,USD,English,0.670000,0.25000,0.420000,Bad
5,The Shawshank Redemption,Hollywood,1994,9.3,Castle Rock Entertainment,25.00,73.3,Millions,USD,English,0.073300,0.02500,0.048300,Excellent
6,Interstellar,Hollywood,2014,8.6,Warner Bros. Pictures,165.00,701.8,Millions,USD,English,0.701800,0.16500,0.536800,Excellent
7,The Pursuit of Happyness,Hollywood,2006,8.0,Columbia Pictures,55.00,307.1,Millions,USD,English,0.307100,0.05500,0.252100,Good
8,Gladiator,Hollywood,2000,8.5,Universal Pictures,103.00,460.5,Millions,USD,English,0.460500,0.10300,0.357500,Excellent
9,Titanic,Hollywood,1997,7.9,Paramount Pictures,200.00,2202.0,Millions,USD,English,2.202000,0.20000,2.002000,Good


## 🔥 MISSÃO 9 — Distribuição

👉 Conte:

quantos filmes existem em cada categoria

In [124]:
print(df.groupby("rating_category")["title"].count())

rating_category
Bad           4
Excellent    20
Good         13
Name: title, dtype: int64


## 🔥 MISSÃO 10 — Insight final

👉 Responda:

“Qual tipo de filme parece mais consistente em qualidade?”

💡 Use:

industry OU language OU categoria

In [134]:
print(df.groupby("industry").agg({
    "imdb_rating": "mean",
    "budget_USD": "mean",
    "revenue_USD": "mean",
    "profit_USD": "mean",
}))

print(df.groupby(["industry", "rating_category"])["title"].count())

# Resultado: A Indústria de Hollywood apresenta ter uma qualidade maior

           imdb_rating  budget_USD  revenue_USD  profit_USD
industry                                                   
Bollywood     7.670588    0.012055     0.052353    0.040298
Hollywood     8.130000    0.153479     0.928895    0.775416
industry   rating_category
Bollywood  Bad                 1
           Excellent           9
           Good                7
Hollywood  Bad                 3
           Excellent          11
           Good                6
Name: title, dtype: int64


In [135]:
print(df.groupby("studio").agg({
    "imdb_rating": "mean",
    "budget_USD": "mean",
    "revenue_USD": "mean",
    "profit_USD": "mean",
}))

print(df.groupby(["studio", "rating_category"])["title"].count())
# 

                           imdb_rating  budget_USD  revenue_USD  profit_USD
studio                                                                     
20th Century Fox              7.800000    0.237000     2.847000    2.610000
Arka Media Works              8.000000    0.019800     0.071500    0.051700
Castle Rock Entertainment     9.300000    0.025000     0.073300    0.048300
Columbia Pictures             8.000000    0.055000     0.307100    0.252100
DVV Entertainment             8.000000    0.060500     0.132000    0.071500
Dharma Productions            7.900000    0.004895     0.012705    0.007810
Government of West Bengal     8.300000    0.000770     0.001100    0.000330
Hombale Films                 8.400000    0.011000     0.137500    0.126500
Liberty Films                 8.600000    0.003180     0.003300    0.000120
Marvel Studios                7.500000    0.248588     1.131825    0.883238
Mythri Movie Makers           7.600000    0.022000     0.039600    0.017600
Paramount Pi